# 00 — Shared Pipeline

Runs once per video, produces a merged per-frame DataFrame shared by all downstream metric notebooks.

**Extractors:**
- **Py-Feat** `Detector` → AUs, emotions, head pose (pitch/yaw/roll)
- **MediaPipe Face Mesh** (with iris refinement) → iris + eye-corner landmarks
- **MediaPipe Pose** → body landmarks (shoulders, wrists, etc.)
- **MediaPipe Hands** → hand landmarks

**Output:** `data/<video_stem>_merged.parquet` indexed by frame, with a `timestamp` column (seconds).

All three MediaPipe models share a single `cv2.VideoCapture` loop for efficiency.

In [2]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from feat import Detector

pd.set_option("display.max_columns", 50)

objc[3979]: Class AVFFrameReceiver is implemented in both /Users/atharvumap/Documents/Projects/PyfeatTesting/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1134d43a8) and /Users/atharvumap/Documents/Projects/PyfeatTesting/.venv/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x3490cc3a8). One of the two will be used. Which one is undefined.
objc[3979]: Class AVFAudioReceiver is implemented in both /Users/atharvumap/Documents/Projects/PyfeatTesting/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1134d43f8) and /Users/atharvumap/Documents/Projects/PyfeatTesting/.venv/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x3490cc3f8). One of the two will be used. Which one is undefined.


## Config

Set `VIDEO_PATH` to the file you want to process. `SKIP_FRAMES=N` processes every Nth frame (1 = every frame). Py-Feat and MediaPipe both honor this so their frame indices stay aligned.

In [3]:
VIDEO_PATH = "../sample.mp4"  # <-- change me
SKIP_FRAMES = 5

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

video_stem = Path(VIDEO_PATH).stem
OUTPUT_PATH = DATA_DIR / f"{video_stem}_merged.parquet"
print(f"Video:  {VIDEO_PATH}")
print(f"Output: {OUTPUT_PATH}")

Video:  ../sample.mp4
Output: /Users/atharvumap/Documents/Projects/PyfeatTesting/data/sample_merged.parquet


In [4]:
# Grab video metadata once
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open {VIDEO_PATH}"
FPS = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"{WIDTH}x{HEIGHT} @ {FPS:.2f} fps, {N_FRAMES} frames ({N_FRAMES/FPS:.1f}s)")

1280x720 @ 30.00 fps, 822 frames (27.4s)


## 1. Py-Feat

In [5]:
detector = Detector()
fex = detector.detect_video(VIDEO_PATH, skip_frames=SKIP_FRAMES)
print(fex.shape)
fex.head(3)

100%|██████████| 165/165 [03:51<00:00,  1.40s/it]


(165, 687)


,FaceRectX,FaceRectY,FaceRectWidth,FaceRectHeight,FaceScore,x_0,x_1,x_2,x_3,x_4,x_5,x_6,x_7,x_8,x_9,x_10,x_11,x_12,x_13,x_14,x_15,x_16,x_17,x_18,x_19,...,Identity_491,Identity_492,Identity_493,Identity_494,Identity_495,Identity_496,Identity_497,Identity_498,Identity_499,Identity_500,Identity_501,Identity_502,Identity_503,Identity_504,Identity_505,Identity_506,Identity_507,Identity_508,Identity_509,Identity_510,Identity_511,Identity_512,input,frame,approx_time
frame,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,519.918694,272.933362,227.425503,281.001893,0.999181,520.523042,523.675914,529.473977,537.559496,548.567274,564.548551,585.259794,609.547189,638.061438,666.781347,691.382727,712.055732,728.102691,738.138415,744.704459,749.427722,751.621890,538.583320,554.554834,575.489435,...,0.002467,-0.010619,0.015837,0.000584,-0.027346,-0.009590,-0.016611,0.026303,0.042407,-0.026899,-0.037780,-0.004625,-0.030934,0.049731,-0.030118,-0.070074,-0.013940,-0.012143,-0.012122,-0.015918,-0.007811,0.008927,../sample.mp4,0,00:00
5,524.518676,274.493958,229.274511,278.821156,0.999232,525.741125,529.029156,534.920038,542.973800,553.899636,569.651581,590.099177,614.189544,642.442699,670.922615,695.329245,715.888461,731.949390,742.251125,749.123409,754.079504,756.550562,543.382180,559.387747,580.751767,...,0.008963,-0.014776,0.021915,0.010424,-0.020274,-0.007171,-0.013455,0.036683,0.050662,-0.027438,-0.035561,0.000410,-0.026599,0.052151,-0.029050,-0.069635,-0.008645,-0.018876,-0.009606,-0.010177,-0.008342,0.018667,../sample.mp4,5,00:00
10,524.179340,272.163873,233.032264,283.197949,0.999275,530.382916,532.495140,537.666197,545.400268,555.938200,571.302844,591.388621,615.529101,644.033582,672.916910,697.468919,718.119562,734.515008,745.391918,752.882629,757.968884,760.156075,546.836208,562.824010,584.130832,...,0.003637,-0.021861,0.027187,0.010508,-0.029896,-0.013428,-0.010573,0.042902,0.049742,-0.027979,-0.038024,-0.000198,-0.029926,0.049220,-0.045106,-0.061819,-0.002862,-0.023451,-0.011430,-0.017750,-0.012166,0.019844,../sample.mp4,10,00:00


In [6]:
# Normalize Py-Feat output to a plain DataFrame with an integer 'frame' column,
# then prefix every non-frame column with 'pf_' so it won't collide with MediaPipe columns on merge.
pf_raw = pd.DataFrame(fex)

# Locate the frame info. Py-Feat can put it in the index (named or unnamed) or as a column.
if "frame" in pf_raw.columns:
    frame_series = pf_raw["frame"]
    pf = pf_raw.drop(columns=["frame"]).reset_index(drop=True)
elif pf_raw.index.name == "frame":
    frame_series = pf_raw.index.to_series().reset_index(drop=True)
    pf = pf_raw.reset_index(drop=True)
else:
    # Fall back: assume the (possibly unnamed) index is the frame counter.
    frame_series = pf_raw.index.to_series().reset_index(drop=True)
    pf = pf_raw.reset_index(drop=True)

pf = pf.add_prefix("pf_")
pf.insert(0, "frame", pd.to_numeric(frame_series.values, errors="coerce"))
pf = pf.dropna(subset=["frame"]).copy()
pf["frame"] = pf["frame"].astype(int)
print(pf.shape)
pf.head(3)


(165, 687)


,frame,pf_FaceRectX,pf_FaceRectY,pf_FaceRectWidth,pf_FaceRectHeight,pf_FaceScore,pf_x_0,pf_x_1,pf_x_2,pf_x_3,pf_x_4,pf_x_5,pf_x_6,pf_x_7,pf_x_8,pf_x_9,pf_x_10,pf_x_11,pf_x_12,pf_x_13,pf_x_14,pf_x_15,pf_x_16,pf_x_17,pf_x_18,...,pf_Identity_490,pf_Identity_491,pf_Identity_492,pf_Identity_493,pf_Identity_494,pf_Identity_495,pf_Identity_496,pf_Identity_497,pf_Identity_498,pf_Identity_499,pf_Identity_500,pf_Identity_501,pf_Identity_502,pf_Identity_503,pf_Identity_504,pf_Identity_505,pf_Identity_506,pf_Identity_507,pf_Identity_508,pf_Identity_509,pf_Identity_510,pf_Identity_511,pf_Identity_512,pf_input,pf_approx_time
0,0,519.918694,272.933362,227.425503,281.001893,0.999181,520.523042,523.675914,529.473977,537.559496,548.567274,564.548551,585.259794,609.547189,638.061438,666.781347,691.382727,712.055732,728.102691,738.138415,744.704459,749.427722,751.621890,538.583320,554.554834,...,-0.017023,0.002467,-0.010619,0.015837,0.000584,-0.027346,-0.009590,-0.016611,0.026303,0.042407,-0.026899,-0.037780,-0.004625,-0.030934,0.049731,-0.030118,-0.070074,-0.013940,-0.012143,-0.012122,-0.015918,-0.007811,0.008927,../sample.mp4,00:00
1,5,524.518676,274.493958,229.274511,278.821156,0.999232,525.741125,529.029156,534.920038,542.973800,553.899636,569.651581,590.099177,614.189544,642.442699,670.922615,695.329245,715.888461,731.949390,742.251125,749.123409,754.079504,756.550562,543.382180,559.387747,...,-0.019737,0.008963,-0.014776,0.021915,0.010424,-0.020274,-0.007171,-0.013455,0.036683,0.050662,-0.027438,-0.035561,0.000410,-0.026599,0.052151,-0.029050,-0.069635,-0.008645,-0.018876,-0.009606,-0.010177,-0.008342,0.018667,../sample.mp4,00:00
2,10,524.179340,272.163873,233.032264,283.197949,0.999275,530.382916,532.495140,537.666197,545.400268,555.938200,571.302844,591.388621,615.529101,644.033582,672.916910,697.468919,718.119562,734.515008,745.391918,752.882629,757.968884,760.156075,546.836208,562.824010,...,-0.014213,0.003637,-0.021861,0.027187,0.010508,-0.029896,-0.013428,-0.010573,0.042902,0.049742,-0.027979,-0.038024,-0.000198,-0.029926,0.049220,-0.045106,-0.061819,-0.002862,-0.023451,-0.011430,-0.017750,-0.012166,0.019844,../sample.mp4,00:00


## 2. MediaPipe (Face Mesh + Pose + Hands)

Single video-capture pass; all three solutions run on each sampled frame.

In [7]:
# Landmark indices we care about.
# Face Mesh iris (requires refine_landmarks=True): 468-472 left, 473-477 right.
# Left iris center: 468, right iris center: 473.
# Eye corners (outer, inner): left 33/133, right 263/362.
FACE_POINTS = {
    "l_iris": 468, "r_iris": 473,
    "l_eye_outer": 33, "l_eye_inner": 133,
    "r_eye_inner": 362, "r_eye_outer": 263,
    "l_eye_top": 159, "l_eye_bot": 145,
    "r_eye_top": 386, "r_eye_bot": 374,
}
# Pose landmarks we care about (MediaPipe Pose indexing).
POSE_POINTS = {
    "nose": 0,
    "l_shoulder": 11, "r_shoulder": 12,
    "l_elbow": 13, "r_elbow": 14,
    "l_wrist": 15, "r_wrist": 16,
    "l_hip": 23, "r_hip": 24,
}

In [8]:
def run_mediapipe(video_path: str, skip_frames: int = 1) -> pd.DataFrame:
    """Return a per-frame DataFrame of MediaPipe landmarks (normalized 0-1 coords)."""
    mp_face = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False, max_num_faces=1,
        refine_landmarks=True, min_detection_confidence=0.5,
    )
    mp_pose = mp.solutions.pose.Pose(
        static_image_mode=False, model_complexity=1, min_detection_confidence=0.5,
    )
    mp_hands = mp.solutions.hands.Hands(
        static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)
    rows = []
    frame_idx = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % skip_frames != 0:
                frame_idx += 1
                continue
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            row = {"frame": frame_idx}

            face_res = mp_face.process(rgb)
            if face_res.multi_face_landmarks:
                lms = face_res.multi_face_landmarks[0].landmark
                for name, idx in FACE_POINTS.items():
                    row[f"mp_face_{name}_x"] = lms[idx].x
                    row[f"mp_face_{name}_y"] = lms[idx].y
                    row[f"mp_face_{name}_z"] = lms[idx].z

            pose_res = mp_pose.process(rgb)
            if pose_res.pose_landmarks:
                lms = pose_res.pose_landmarks.landmark
                for name, idx in POSE_POINTS.items():
                    row[f"mp_pose_{name}_x"] = lms[idx].x
                    row[f"mp_pose_{name}_y"] = lms[idx].y
                    row[f"mp_pose_{name}_z"] = lms[idx].z
                    row[f"mp_pose_{name}_v"] = lms[idx].visibility

            hands_res = mp_hands.process(rgb)
            if hands_res.multi_hand_landmarks and hands_res.multi_handedness:
                for hlm, handed in zip(hands_res.multi_hand_landmarks, hands_res.multi_handedness):
                    label = handed.classification[0].label.lower()  # 'left' or 'right'
                    for i, lm in enumerate(hlm.landmark):
                        row[f"mp_hand_{label}_{i}_x"] = lm.x
                        row[f"mp_hand_{label}_{i}_y"] = lm.y
                        row[f"mp_hand_{label}_{i}_z"] = lm.z

            rows.append(row)
            frame_idx += 1
    finally:
        cap.release()
        mp_face.close()
        mp_pose.close()
        mp_hands.close()
    return pd.DataFrame(rows)

In [9]:
mp_df = run_mediapipe(VIDEO_PATH, skip_frames=SKIP_FRAMES)
print(mp_df.shape)
mp_df.head(3)

I0000 00:00:1776437570.245425   70496 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776437570.247732   77821 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776437570.250482   77821 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776437570.250754   70496 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4 Pro
I0000 00:00:1776437570.254071   70496 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4 Pro
W0000 00:00:1776437570.258856   77850 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776437570.262484   77850 inference_fe

(165, 67)


,frame,mp_face_l_iris_x,mp_face_l_iris_y,mp_face_l_iris_z,mp_face_r_iris_x,mp_face_r_iris_y,mp_face_r_iris_z,mp_face_l_eye_outer_x,mp_face_l_eye_outer_y,mp_face_l_eye_outer_z,mp_face_l_eye_inner_x,mp_face_l_eye_inner_y,mp_face_l_eye_inner_z,mp_face_r_eye_inner_x,mp_face_r_eye_inner_y,mp_face_r_eye_inner_z,mp_face_r_eye_outer_x,mp_face_r_eye_outer_y,mp_face_r_eye_outer_z,mp_face_l_eye_top_x,mp_face_l_eye_top_y,mp_face_l_eye_top_z,mp_face_l_eye_bot_x,mp_face_l_eye_bot_y,mp_face_l_eye_bot_z,...,mp_pose_r_shoulder_v,mp_pose_l_elbow_x,mp_pose_l_elbow_y,mp_pose_l_elbow_z,mp_pose_l_elbow_v,mp_pose_r_elbow_x,mp_pose_r_elbow_y,mp_pose_r_elbow_z,mp_pose_r_elbow_v,mp_pose_l_wrist_x,mp_pose_l_wrist_y,mp_pose_l_wrist_z,mp_pose_l_wrist_v,mp_pose_r_wrist_x,mp_pose_r_wrist_y,mp_pose_r_wrist_z,mp_pose_r_wrist_v,mp_pose_l_hip_x,mp_pose_l_hip_y,mp_pose_l_hip_z,mp_pose_l_hip_v,mp_pose_r_hip_x,mp_pose_r_hip_y,mp_pose_r_hip_z,mp_pose_r_hip_v
0,0,0.456598,0.529413,0.000589,0.538816,0.526038,-0.000963,0.440425,0.530901,0.007921,0.476623,0.532623,0.000656,0.520675,0.532937,-0.000382,0.554804,0.527530,0.005815,0.453742,0.522118,-0.003715,0.457940,0.541301,-0.000848,...,0.998591,0.788142,1.078022,-0.256202,0.284957,0.195515,1.240849,-0.312661,0.182088,0.826058,1.551448,-0.424262,0.094261,0.203969,1.569180,-0.680746,0.055634,0.617857,1.597938,-0.026680,0.000262,0.383883,1.642882,0.031453,0.000211
1,5,0.459855,0.530209,0.000793,0.542481,0.527377,-0.000243,0.443913,0.531753,0.008206,0.479350,0.533371,0.000873,0.524215,0.532599,0.000212,0.559125,0.528815,0.006809,0.457282,0.522721,-0.003510,0.460714,0.541014,-0.000726,...,0.998580,0.788900,1.072062,-0.274050,0.285622,0.177528,1.237659,-0.382536,0.200346,0.838439,1.560048,-0.490155,0.089535,0.155611,1.569308,-0.717756,0.054093,0.617698,1.631448,-0.027086,0.000271,0.383881,1.648865,0.031783,0.000213
2,10,0.461720,0.531697,0.000815,0.544931,0.530403,-0.000533,0.445766,0.533398,0.008216,0.480774,0.535768,0.000835,0.526106,0.535791,-0.000010,0.561113,0.532016,0.006388,0.458893,0.524873,-0.003446,0.462767,0.543541,-0.000689,...,0.998564,0.794182,1.202609,-0.234528,0.281576,0.175390,1.247655,-0.377758,0.213823,0.838154,1.579179,-0.486390,0.085271,0.159465,1.584551,-0.711673,0.052361,0.619557,1.653885,-0.019115,0.000287,0.385019,1.665081,0.023625,0.000223


## 3. Merge and save

In [10]:
merged = pf.merge(mp_df, on="frame", how="outer").sort_values("frame").reset_index(drop=True)
merged["timestamp"] = merged["frame"] / FPS

# Attach video-level metadata as parquet attrs via a sidecar dict (store as string col if needed downstream).
print(f"Merged shape: {merged.shape}")
print(f"Frames: {merged['frame'].min()} → {merged['frame'].max()}")
print(f"Duration covered: {merged['timestamp'].max():.2f}s")
merged.head(3)

Merged shape: (165, 754)
Frames: 0 → 820
Duration covered: 27.33s


,frame,pf_FaceRectX,pf_FaceRectY,pf_FaceRectWidth,pf_FaceRectHeight,pf_FaceScore,pf_x_0,pf_x_1,pf_x_2,pf_x_3,pf_x_4,pf_x_5,pf_x_6,pf_x_7,pf_x_8,pf_x_9,pf_x_10,pf_x_11,pf_x_12,pf_x_13,pf_x_14,pf_x_15,pf_x_16,pf_x_17,pf_x_18,...,mp_pose_l_elbow_x,mp_pose_l_elbow_y,mp_pose_l_elbow_z,mp_pose_l_elbow_v,mp_pose_r_elbow_x,mp_pose_r_elbow_y,mp_pose_r_elbow_z,mp_pose_r_elbow_v,mp_pose_l_wrist_x,mp_pose_l_wrist_y,mp_pose_l_wrist_z,mp_pose_l_wrist_v,mp_pose_r_wrist_x,mp_pose_r_wrist_y,mp_pose_r_wrist_z,mp_pose_r_wrist_v,mp_pose_l_hip_x,mp_pose_l_hip_y,mp_pose_l_hip_z,mp_pose_l_hip_v,mp_pose_r_hip_x,mp_pose_r_hip_y,mp_pose_r_hip_z,mp_pose_r_hip_v,timestamp
0,0,519.918694,272.933362,227.425503,281.001893,0.999181,520.523042,523.675914,529.473977,537.559496,548.567274,564.548551,585.259794,609.547189,638.061438,666.781347,691.382727,712.055732,728.102691,738.138415,744.704459,749.427722,751.621890,538.583320,554.554834,...,0.788142,1.078022,-0.256202,0.284957,0.195515,1.240849,-0.312661,0.182088,0.826058,1.551448,-0.424262,0.094261,0.203969,1.569180,-0.680746,0.055634,0.617857,1.597938,-0.026680,0.000262,0.383883,1.642882,0.031453,0.000211,0.000000
1,5,524.518676,274.493958,229.274511,278.821156,0.999232,525.741125,529.029156,534.920038,542.973800,553.899636,569.651581,590.099177,614.189544,642.442699,670.922615,695.329245,715.888461,731.949390,742.251125,749.123409,754.079504,756.550562,543.382180,559.387747,...,0.788900,1.072062,-0.274050,0.285622,0.177528,1.237659,-0.382536,0.200346,0.838439,1.560048,-0.490155,0.089535,0.155611,1.569308,-0.717756,0.054093,0.617698,1.631448,-0.027086,0.000271,0.383881,1.648865,0.031783,0.000213,0.166667
2,10,524.179340,272.163873,233.032264,283.197949,0.999275,530.382916,532.495140,537.666197,545.400268,555.938200,571.302844,591.388621,615.529101,644.033582,672.916910,697.468919,718.119562,734.515008,745.391918,752.882629,757.968884,760.156075,546.836208,562.824010,...,0.794182,1.202609,-0.234528,0.281576,0.175390,1.247655,-0.377758,0.213823,0.838154,1.579179,-0.486390,0.085271,0.159465,1.584551,-0.711673,0.052361,0.619557,1.653885,-0.019115,0.000287,0.385019,1.665081,0.023625,0.000223,0.333333


In [11]:
merged.to_parquet(OUTPUT_PATH, index=False)

# Write a small sidecar with video metadata so downstream notebooks know FPS etc.
meta = {
    "video_path": str(Path(VIDEO_PATH).resolve()),
    "fps": float(FPS),
    "n_frames": int(N_FRAMES),
    "width": int(WIDTH),
    "height": int(HEIGHT),
    "skip_frames": int(SKIP_FRAMES),
    "effective_fps": float(FPS / SKIP_FRAMES),
}
pd.Series(meta).to_json(OUTPUT_PATH.with_suffix(".meta.json"))
print(f"Saved: {OUTPUT_PATH}")
print(f"Saved: {OUTPUT_PATH.with_suffix('.meta.json')}")

Saved: /Users/atharvumap/Documents/Projects/PyfeatTesting/data/sample_merged.parquet
Saved: /Users/atharvumap/Documents/Projects/PyfeatTesting/data/sample_merged.meta.json


## Sanity check

Quick look at what's in the merged frame. Downstream notebooks should start by loading this parquet + its sidecar json.

In [12]:
loaded = pd.read_parquet(OUTPUT_PATH)
meta_loaded = pd.read_json(OUTPUT_PATH.with_suffix(".meta.json"), typ="series")
print("meta:", dict(meta_loaded))
print("columns (sample):", [c for c in loaded.columns[:20]])
print("n rows:", len(loaded))

meta: {'video_path': '/Users/atharvumap/Documents/Projects/PyfeatTesting/sample.mp4', 'fps': 30.0, 'n_frames': 822, 'width': 1280, 'height': 720, 'skip_frames': 5, 'effective_fps': 6.0}
columns (sample): ['frame', 'pf_FaceRectX', 'pf_FaceRectY', 'pf_FaceRectWidth', 'pf_FaceRectHeight', 'pf_FaceScore', 'pf_x_0', 'pf_x_1', 'pf_x_2', 'pf_x_3', 'pf_x_4', 'pf_x_5', 'pf_x_6', 'pf_x_7', 'pf_x_8', 'pf_x_9', 'pf_x_10', 'pf_x_11', 'pf_x_12', 'pf_x_13']
n rows: 165
